In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
tables_dir = Path("../../results/data_processing/tables")
df = pd.read_csv("../../data/processed/real_estate_data_updated.csv")

target = "Y house price of unit area"

numerical_features = [
    "X2 house age",
    "X3 distance to the nearest MRT station",
    "X4 number of convenience stores",
    "X5 latitude",
    "X6 longitude"
]

categorical_features = [
    "X1 transaction year",
    "X1 transaction month"
]

x = pd.get_dummies(
    df[numerical_features + categorical_features],
    columns=categorical_features,
    drop_first=True,
    dtype=float
)

x = x.loc[:, x.nunique() > 1]

x_array = x.to_numpy(dtype=float)

print("Design matrix shape:", x_array.shape)
print("Features:", x.columns.tolist())

Design matrix shape: (414, 17)
Features: ['X2 house age', 'X3 distance to the nearest MRT station', 'X4 number of convenience stores', 'X5 latitude', 'X6 longitude', 'X1 transaction year_2013', 'X1 transaction month_August', 'X1 transaction month_December', 'X1 transaction month_February', 'X1 transaction month_January', 'X1 transaction month_July', 'X1 transaction month_June', 'X1 transaction month_March', 'X1 transaction month_May', 'X1 transaction month_November', 'X1 transaction month_October', 'X1 transaction month_September']


In [3]:
print("\n[1] condition number - raw matrix")

condition_raw = np.linalg.cond(x_array)
print("condition number", condition_raw)


[1] condition number - raw matrix
condition number 53306857.76649755


In [4]:
scaler = StandardScaler()

x_scaled = scaler.fit_transform(x_array)

In [5]:
# condition number - standardized matrix

print("\n[2] condition number - standardized matrix")

condition_scaled = np.linalg.cond(x_scaled)

print("condition number:", condition_scaled)


[2] condition number - standardized matrix
condition number: 7781421702636002.0


In [6]:
# condition nummber of the x transpose x

print("\n[3] condition number of x transpose x")

condition_xtx = np.linalg.cond(x_array.T @ x_array)

condition_xtx_scaled = np.linalg.cond(
    x_scaled.T @ x_scaled
)

print("raw x.T @ x:", condition_xtx)
print("standardized x.T @ x", condition_xtx_scaled)


[3] condition number of x transpose x
raw x.T @ x: 2841621147444542.0
standardized x.T @ x 4300832655510737.0


In [9]:
# vaiance inflation factor

print("\n[4] variance inflation factor")

vif_data = pd.DataFrame({
    "Feature": x.columns,
    "VIF": [
        variance_inflation_factor(x_scaled, i)
        for i in range(x_scaled.shape[1])
    ]
})

vif_data = vif_data.sort_values(
    by="VIF",
    ascending=False
)

print(vif_data)

vif_data.to_csv(
    tables_dir / "vif_results.csv",
    index=False
)


[4] variance inflation factor
                                   Feature           VIF
7            X1 transaction month_December  1.000800e+15
14           X1 transaction month_November  1.000800e+15
5                 X1 transaction year_2013  1.000800e+15
16          X1 transaction month_September  1.000800e+15
15            X1 transaction month_October  1.000800e+15
1   X3 distance to the nearest MRT station  4.528575e+00
4                             X6 longitude  3.043901e+00
11               X1 transaction month_June  2.441750e+00
10               X1 transaction month_July  2.265422e+00
8            X1 transaction month_February  2.186006e+00
9             X1 transaction month_January  1.845525e+00
13                X1 transaction month_May  1.781479e+00
12              X1 transaction month_March  1.678679e+00
2          X4 number of convenience stores  1.656715e+00
3                              X5 latitude  1.645398e+00
6              X1 transaction month_August  1.644984e+00


C:\Users\Admin\AppData\Local\Temp\ipykernel_11032\1256043054.py:8: UserWarning: The design matrix is poorly conditioned (condition number=4.33e+15). VIF calculations may be numerically unstable.
  variance_inflation_factor(x_scaled, i)
c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\stats\outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\stats\outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\stats\outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined

In [8]:
# summary

print("raw x:", condition_raw)

print("standardized x:", condition_scaled)

print("raw x.T @ x:", condition_xtx)

print("standardized x.T @ x:", condition_xtx_scaled)

condition_summary = pd.DataFrame({
    "Matrix": [
        "Raw X",
        "Standardized X",
        "Raw X.T @ X",
        "Standardized X.T @ X"
    ],
    "Condition Number": [
        condition_raw,
        condition_scaled,
        condition_xtx,
        condition_xtx_scaled
    ]
})

condition_summary.to_csv(
    tables_dir/"condition_number_summary.csv",
    index=False
)

raw x: 53306857.76649755
standardized x: 7781421702636002.0
raw x.T @ x: 2841621147444542.0
standardized x.T @ x: 4300832655510737.0
